# BA820 Project — M 4: Refining Exit Pathways via Association Rules

## Research Focus
**Question:** Do recognizable “exit pathways” exist among *Alone* contestants, and are these patterns robust to modeling choices (thresholds, encoding decisions, sampling/season splits)?

## What changed from M2/M3
- M2 established interpretable exit-pathway patterns using Apriori rules on a transaction representation.
- M3 integrated my pathway labels with strategy clusters (team synergy).
- **M4 improves rigor:** compare alternative mining algorithm(s), run threshold sensitivity, and test stability/robustness across resamples or season splits.

##Imports

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from mlxtend.frequent_patterns import apriori, association_rules

import warnings
warnings.filterwarnings("ignore")

##Data Loading

In [2]:
BASE_URL = "https://raw.githubusercontent.com/Marcusshi/BA820-A1-08/main/data/alone_tv_show/"
survivalists = pd.read_csv(BASE_URL + "survivalists.csv")
loadouts = pd.read_csv(BASE_URL + "loadouts.csv")
episodes = pd.read_csv(BASE_URL + "episodes.csv")

print("survivalists:", survivalists.shape)
print("episodes:", episodes.shape)
print("loadouts:", loadouts.shape)

survivalists.head()

survivalists: (94, 16)
episodes: (98, 11)
loadouts: (940, 6)


,season,name,age,gender,city,state,country,result,days_lasted,medically_evacuated,reason_tapped_out,reason_category,team,day_linked_up,profession,url
0,1,Alan Kay,40,Male,Blairsville,Georgia,United States,1,56,False,NaN,NaN,NaN,NaN,Corrections Officer,alan-kay
1,1,Sam Larson,22,Male,Lincoln,Nebraska,United States,2,55,False,Lost the mind game,Family / personal,NaN,NaN,Outdoor Gear Retailer,sam-larson
2,1,Mitch Mitchell,34,Male,Bellingham,Massachusetts,United States,3,43,False,Realized he should actually be around for his ...,Family / personal,NaN,NaN,Butcher,mitch-mitchell
3,1,Lucas Miller,32,Male,Quasqueton,Iowa,United States,4,39,False,Felt content with what he had done,Family / personal,NaN,NaN,Survivalist and Wildlife Therapist/Natural Hea...,lucas-miller
4,1,Dustin Feher,37,Male,Pittsburgh,Pennsylvania,United States,5,8,False,Fear of storm,Family / personal,NaN,NaN,Carpenter,dustin-feher


## Minimal Cleaning / Column Sanity Check

Goal: ensure we have the minimum fields needed for pathway mining:
- unique participant identifier
- survival duration (days lasted)
- exit reason / tap-out reason
- season (if available) for robustness checks by season

In [3]:
survivalists.columns

Index(['season', 'name', 'age', 'gender', 'city', 'state', 'country', 'result',
       'days_lasted', 'medically_evacuated', 'reason_tapped_out',
       'reason_category', 'team', 'day_linked_up', 'profession', 'url'],
      dtype='object')

In [4]:
# Correct column names based on actual dataset
SEASON_COL = "season"
NAME_COL = "name"
DAYS_COL = "days_lasted"
REASON_FINE_COL = "reason_tapped_out"
REASON_CAT_COL = "reason_category"

# Create a stable participant ID
df = survivalists.copy()

df["participant_id"] = (
    df[SEASON_COL].astype(str) + "_" +
    df[NAME_COL].astype(str).str.strip().str.lower()
)

# Ensure numeric days
df[DAYS_COL] = pd.to_numeric(df[DAYS_COL], errors="coerce")

# Standardize text columns
for c in [REASON_FINE_COL, REASON_CAT_COL]:
    df[c] = df[c].astype(str).str.strip().str.lower()

# Drop rows missing essentials
df = df.dropna(subset=[DAYS_COL, REASON_FINE_COL, SEASON_COL, "participant_id"]).copy()

print("After cleaning:", df.shape)
df[["participant_id", SEASON_COL, NAME_COL, DAYS_COL, REASON_FINE_COL, REASON_CAT_COL]].head()

After cleaning: (94, 17)


,participant_id,season,name,days_lasted,reason_tapped_out,reason_category
0,1_alan kay,1,Alan Kay,56,nan,nan
1,1_sam larson,1,Sam Larson,55,lost the mind game,family / personal
2,1_mitch mitchell,1,Mitch Mitchell,43,realized he should actually be around for his ...,family / personal
3,1_lucas miller,1,Lucas Miller,39,felt content with what he had done,family / personal
4,1_dustin feher,1,Dustin Feher,8,fear of storm,family / personal


##Transaction Design (Baseline: reason_category)

In M4, I define each contestant as a "transaction" consisting of:

- Exit reason category (`reason_category`)
- Survival timing bucket (derived from `days_lasted`)

This follows the M2 logic but is now explicitly documented and structured for robustness testing.

Using `reason_category` (instead of the more detailed `reason_tapped_out`) improves interpretability and reduces noise in rule mining.

###Create Timing Buckets

In [5]:
# Create timing bins (terciles based on survival duration)
df["timing_bin"] = pd.qcut(
    df[DAYS_COL],
    q=3,
    labels=["early", "mid", "late"]
)

df[["participant_id", DAYS_COL, "timing_bin"]].head()

,participant_id,days_lasted,timing_bin
0,1_alan kay,56,late
1,1_sam larson,55,late
2,1_mitch mitchell,43,mid
3,1_lucas miller,39,mid
4,1_dustin feher,8,early


In [6]:
# Properly handle missing reason_category

df[REASON_CAT_COL] = df[REASON_CAT_COL].replace("nan", np.nan)

df = df.dropna(subset=[REASON_CAT_COL]).copy()

print("After removing missing reason_category:", df.shape)

After removing missing reason_category: (84, 18)


###Build Transaction Items

In [7]:
# Create item labels
df["item_reason"] = "reason=" + df[REASON_CAT_COL]
df["item_timing"] = "timing=" + df["timing_bin"].astype(str)

transactions = df[["participant_id", "item_reason", "item_timing"]].copy()
transactions.head()

,participant_id,item_reason,item_timing
1,1_sam larson,reason=family / personal,timing=late
2,1_mitch mitchell,reason=family / personal,timing=mid
3,1_lucas miller,reason=family / personal,timing=mid
4,1_dustin feher,reason=family / personal,timing=early
5,1_brant mcgee,reason=medical / health,timing=early


###One-Hot Encode Basket

In [8]:
basket = (
    transactions
    .melt(
        id_vars="participant_id",
        value_vars=["item_reason", "item_timing"],
        value_name="item"
    )
    .dropna(subset=["item"])
)

onehot = (
    basket
    .assign(val=1)
    .pivot_table(
        index="participant_id",
        columns="item",
        values="val",
        aggfunc="max",
        fill_value=0
    )
    .astype(bool)
)

print("One-hot shape:", onehot.shape)
onehot.head()

One-hot shape: (84, 6)


item,reason=family / personal,reason=loss of inventory,reason=medical / health,timing=early,timing=late,timing=mid
participant_id,,,,,,
1_brant mcgee,False,False,True,True,False,False
1_chris weatherman,True,False,False,True,False,False
1_dustin feher,True,False,False,True,False,False
1_joe robinet,False,True,False,True,False,False
1_josh chavez,True,False,False,True,False,False


##Baseline Association Rules (Apriori)

This reproduces the core M2 approach.  
In M4, this serves as the baseline model before:

- threshold sensitivity analysis
- algorithm comparison (FP-Growth if available)
- bootstrap / season robustness checks

In [9]:
MIN_SUPPORT = 0.05
MIN_CONFIDENCE = 0.30

freq = apriori(
    onehot,
    min_support=MIN_SUPPORT,
    use_colnames=True
)

rules = association_rules(
    freq,
    metric="confidence",
    min_threshold=MIN_CONFIDENCE
)

# Clean up rule display
rules["antecedents_str"] = rules["antecedents"].apply(lambda s: ", ".join(sorted(list(s))))
rules["consequents_str"] = rules["consequents"].apply(lambda s: ", ".join(sorted(list(s))))

rules_out = rules[[
    "antecedents_str",
    "consequents_str",
    "support",
    "confidence",
    "lift"
]].sort_values(["lift", "confidence", "support"], ascending=False)

rules_out.head(15)

,antecedents_str,consequents_str,support,confidence,lift
7,timing=late,reason=medical / health,0.166667,0.636364,1.187879
8,reason=medical / health,timing=late,0.166667,0.311111,1.187879
4,timing=mid,reason=family / personal,0.178571,0.483871,1.129032
3,reason=family / personal,timing=mid,0.178571,0.416667,1.129032
1,timing=early,reason=family / personal,0.166667,0.451613,1.053763
0,reason=family / personal,timing=early,0.166667,0.388889,1.053763
9,timing=mid,reason=medical / health,0.190476,0.516129,0.963441
10,reason=medical / health,timing=mid,0.190476,0.355556,0.963441
5,reason=medical / health,timing=early,0.178571,0.333333,0.903226
6,timing=early,reason=medical / health,0.178571,0.483871,0.903226


## Threshold Sensitivity Analysis

To evaluate robustness, I systematically vary minimum support and confidence thresholds and observe how:

- number of rules
- maximum lift
- average lift

change across parameter choices.

In [11]:
# Threshold sensitivity grid

support_grid = [0.03, 0.05, 0.08]
confidence_grid = [0.25, 0.30, 0.40]

results = []

for s in support_grid:
    for c in confidence_grid:

        freq_tmp = apriori(onehot, min_support=s, use_colnames=True)
        rules_tmp = association_rules(freq_tmp, metric="confidence", min_threshold=c)

        if len(rules_tmp) > 0:
            results.append({
                "min_support": s,
                "min_confidence": c,
                "n_rules": len(rules_tmp),
                "max_lift": rules_tmp["lift"].max(),
                "avg_lift": rules_tmp["lift"].mean()
            })
        else:
            results.append({
                "min_support": s,
                "min_confidence": c,
                "n_rules": 0,
                "max_lift": np.nan,
                "avg_lift": np.nan
            })

sensitivity_df = pd.DataFrame(results)
sensitivity_df

,min_support,min_confidence,n_rules,max_lift,avg_lift
0,0.03,0.25,11,1.187879,1.019737
1,0.03,0.30,11,1.187879,1.019737
2,0.03,0.40,6,1.187879,1.061062
3,0.05,0.25,11,1.187879,1.019737
4,0.05,0.30,11,1.187879,1.019737
5,0.05,0.40,6,1.187879,1.061062
6,0.08,0.25,11,1.187879,1.019737
7,0.08,0.30,11,1.187879,1.019737
8,0.08,0.40,6,1.187879,1.061062


**Interpretation of Threshold Sensitivity**

The strongest exit pathway (late timing → medical/health) remains stable across all tested support (0.03–0.08) and confidence (0.25–0.40) thresholds.

- Maximum lift remains constant at 1.19.
- Increasing confidence reduces weaker rules but does not eliminate the core pathway.
- Average lift increases slightly as weaker rules are filtered out.

This suggests that the identified exit structure is not an artifact of arbitrary parameter choices, but reflects a stable association pattern within the dataset.

In [12]:
from mlxtend.frequent_patterns import fpgrowth

In [13]:
# FP-Growth comparison

freq_fp = fpgrowth(
    onehot,
    min_support=0.05,
    use_colnames=True
)

rules_fp = association_rules(
    freq_fp,
    metric="confidence",
    min_threshold=0.30
)

rules_fp["antecedents_str"] = rules_fp["antecedents"].apply(lambda s: ", ".join(sorted(list(s))))
rules_fp["consequents_str"] = rules_fp["consequents"].apply(lambda s: ", ".join(sorted(list(s))))

rules_fp_out = rules_fp[[
    "antecedents_str",
    "consequents_str",
    "support",
    "confidence",
    "lift"
]].sort_values(["lift", "confidence"], ascending=False)

rules_fp_out.head(10)

,antecedents_str,consequents_str,support,confidence,lift
9,timing=late,reason=medical / health,0.166667,0.636364,1.187879
10,reason=medical / health,timing=late,0.166667,0.311111,1.187879
5,timing=mid,reason=family / personal,0.178571,0.483871,1.129032
4,reason=family / personal,timing=mid,0.178571,0.416667,1.129032
3,timing=early,reason=family / personal,0.166667,0.451613,1.053763
2,reason=family / personal,timing=early,0.166667,0.388889,1.053763
6,timing=mid,reason=medical / health,0.190476,0.516129,0.963441
7,reason=medical / health,timing=mid,0.190476,0.355556,0.963441
0,reason=medical / health,timing=early,0.178571,0.333333,0.903226
1,timing=early,reason=medical / health,0.178571,0.483871,0.903226


**Refined Interpretation of Exit Pathways**

The late → medical pathway is stable with respect to algorithm choice and threshold variation.
However, season-based analysis reveals that the timing concentration of medical exits shifts across eras, indicating temporal heterogeneity.

This rule remains consistent across:

* Support thresholds (0.03–0.08)

* Confidence thresholds (0.25–0.40)

* Apriori and FP-Growth algorithms

The lift of ~1.19 indicates a modest but consistent elevation above random expectation.

This suggests that prolonged exposure increases the probability of medically driven exits, rather than early-stage attrition.

## Season-Based Robustness Test

To evaluate structural stability across time, I split the dataset into:

- Early seasons
- Later seasons

I then re-run the association rule mining process separately on each subset to test whether the dominant pathway (late → medical/health) persists.

If the rule appears in both splits with similar lift values, this suggests temporal robustness.

###Define Season Split

In [14]:
# Determine median season for split
median_season = df[SEASON_COL].median()
median_season

5.0

###Create Subsets

In [15]:
# Early vs Later seasons

df_early = df[df[SEASON_COL] <= median_season].copy()
df_late = df[df[SEASON_COL] > median_season].copy()

print("Early seasons shape:", df_early.shape)
print("Later seasons shape:", df_late.shape)

Early seasons shape: (48, 20)
Later seasons shape: (36, 20)


###Function to Rebuild Rules

In [16]:
def run_rules(data_subset, min_support=0.05, min_confidence=0.30):

    # Recreate timing bins within subset
    data_subset = data_subset.copy()
    data_subset["timing_bin"] = pd.qcut(
        data_subset[DAYS_COL],
        q=3,
        labels=["early", "mid", "late"]
    )

    data_subset["item_reason"] = "reason=" + data_subset[REASON_CAT_COL]
    data_subset["item_timing"] = "timing=" + data_subset["timing_bin"].astype(str)

    transactions = data_subset[["participant_id", "item_reason", "item_timing"]]

    basket = (
        transactions
        .melt(id_vars="participant_id",
              value_vars=["item_reason", "item_timing"],
              value_name="item")
        .dropna(subset=["item"])
    )

    onehot = (
        basket.assign(val=1)
        .pivot_table(index="participant_id",
                     columns="item",
                     values="val",
                     aggfunc="max",
                     fill_value=0)
        .astype(bool)
    )

    freq = apriori(onehot, min_support=min_support, use_colnames=True)
    rules = association_rules(freq, metric="confidence", min_threshold=min_confidence)

    rules["antecedents_str"] = rules["antecedents"].apply(lambda s: ", ".join(sorted(list(s))))
    rules["consequents_str"] = rules["consequents"].apply(lambda s: ", ".join(sorted(list(s))))

    return rules.sort_values(["lift", "confidence"], ascending=False)

Timing bins are recomputed within each subset to preserve relative survival positioning inside each season group.

###Run for Early Seasons

In [17]:
rules_early = run_rules(df_early)

rules_early[[
    "antecedents_str",
    "consequents_str",
    "support",
    "confidence",
    "lift"
]].head(10)

,antecedents_str,consequents_str,support,confidence,lift
6,timing=early,reason=medical / health,0.166667,0.500000,1.411765
5,reason=medical / health,timing=early,0.166667,0.470588,1.411765
4,timing=mid,reason=family / personal,0.270833,0.812500,1.300000
3,reason=family / personal,timing=mid,0.270833,0.433333,1.300000
7,timing=late,reason=medical / health,0.125000,0.375000,1.058824
8,reason=medical / health,timing=late,0.125000,0.352941,1.058824
2,timing=late,reason=family / personal,0.208333,0.625000,1.000000
1,reason=family / personal,timing=late,0.208333,0.333333,1.000000
0,timing=early,reason=family / personal,0.145833,0.437500,0.700000


###Run for Later Seasons

In [18]:
rules_late = run_rules(df_late)

rules_late[[
    "antecedents_str",
    "consequents_str",
    "support",
    "confidence",
    "lift"
]].head(10)

,antecedents_str,consequents_str,support,confidence,lift
0,reason=family / personal,timing=early,0.111111,0.666667,2.000000
1,timing=early,reason=family / personal,0.111111,0.333333,2.000000
5,timing=mid,reason=medical / health,0.333333,0.923077,1.186813
6,reason=medical / health,timing=mid,0.333333,0.428571,1.186813
3,timing=late,reason=medical / health,0.250000,0.818182,1.051948
4,reason=medical / health,timing=late,0.250000,0.321429,1.051948
2,timing=early,reason=medical / health,0.194444,0.583333,0.750000


**Interpretation of Season-Based Robustness**

The dominant pathway identified in the full dataset (late → medical/health) does not remain structurally identical across season splits.

Early seasons:
- Medical exits are overrepresented among early timing (lift ≈ 1.41).
- Mid-season family/personal exits are also strong (lift ≈ 1.30).

Later seasons:
- Medical exits are more concentrated in mid timing (lift ≈ 1.19).
- A high-lift early-family pattern appears (lift ≈ 2.00), though with lower support.

This indicates that exit pathways are not fully time-invariant.  
Instead, structural associations appear to shift across seasons, suggesting temporal heterogeneity in exit dynamics.

## Practical Implications

The analysis suggests that exit pathways are structured but not time-invariant.

While medical exits consistently form a meaningful pathway, the timing concentration shifts across seasons.

This implies that:

- Exit dynamics may evolve due to production changes, environmental conditions, or contestant preparation.
- Models trained on pooled data may obscure season-specific structure.
- Future analysis should consider season-stratified modeling rather than assuming stationary behavior.

Therefore, pathway identification should be treated as context-dependent rather than universal.